# 02. LDA 토픽 모델 — 세계 전체(LLM 에이전트 논문)에서 학습, 한중일어 대상 집합 투영 (2023-01 ~ 2026-09)

- **목적**: 확률 토픽 모델 LDA(Latent Dirichlet Allocation)로 LLM 에이전트 연구의 주제 구조를 뽑고, 그 위에 한국어·중국어·일본어 대상 논문을 투영해 세계 전체 분포와 비교한다(논문 4장). D30에 따라 BERTopic 대신 LDA가 주 모델이다.
- **참고 코드**: pytextbook 7.1 `seoul-120-LDA.ipynb` — `CountVectorizer` → `sklearn.decomposition.LatentDirichletAllocation(random_state=42)` → `pyLDAvis`. 이 노트북은 그 틀을 따르고 다음을 더한다
  - 토큰: spaCy `en_core_web_sm` 표제어(lemma, D26). 영어 불용어 + 도메인 불용어 제거, 세 글자 이상 알파벳 토큰
  - 토픽 수 K: 참고 코드는 분류 수(10)를 썼으나 이 자료에는 정답 분류가 없어 K ∈ {10, 15, 20, 25, 30}을 c_v coherence(gensim)와 보류 집합 perplexity로 비교해 고른다
  - 문서 배정: 문서-토픽 확률의 최대 토픽(argmax). 토픽 라벨은 상위 두 단어(모델 출력, D20)
- **입력**: `preprocessing/output/corpus_llm_agent_all_2023_2026q3.csv`의 제목+초록. 표제어 캐시 `analysis/output/cache/lemmas_2023_2026q3.jsonl`
- **출력**: `verification/topics_lda_YYYYMMDD.json`(정본), `analysis/output/tables/lda_*.csv`, `analysis/output/figures/fig2*_lda_*.png`, `lda_pyldavis_K*.html`
- **주의**
  - 투영 집합의 토픽별 건수가 10건 미만이면 lift를 보고하지 않고 "관측 n편 / 기대 m편"으로만 적는다
  - 대상 언어 라벨은 사람 검수 전이다. 검수 뒤 라벨이 바뀌면 이 노트북을 다시 실행한다


In [ ]:
import glob, json, re, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "verification").exists())   # 노트북은 analysis/notebooks/ 에 있다
DATA = ROOT / "preprocessing/output"                                   # 입력: 전처리 산출(정제 CSV)
VER = ROOT / "verification"                                            # 수치의 정본 JSON
FIG = ROOT / "analysis/output/figures"; FIG.mkdir(parents=True, exist_ok=True)   # 출력: 노트북 그림. 원고 그림은 paper/figures/
TAB = ROOT / "analysis/output/tables"; TAB.mkdir(parents=True, exist_ok=True)    # 출력: 노트북 표
CACHE = ROOT / "analysis/output/cache"; CACHE.mkdir(parents=True, exist_ok=True) # 출력: 재실행용 캐시(git 제외)
TODAY = pd.Timestamp.today().strftime("%Y%m%d")
SUFFIX = "2023_2026q3"   # D30: 분석 기간 2023-01 ~ 2026-09 말뭉치

try:
    sys.path.insert(0, str(ROOT / "analysis/scripts"))
    import figure_style; figure_style.apply_style(); C = figure_style.COLORS; CYC = figure_style.COLOR_CYCLE
except Exception:
    C = {"green": "#46B07B", "orange": "#FF9E02", "teal": "#3A9E8F", "coral": "#E87B4A", "green_dark": "#2D7A53", "orange_dark": "#C47800", "green_light": "#8DD4AA", "orange_light": "#FFD080", "green_pale": "#C8EBDA", "orange_pale": "#FFECC0", "gray": "#7F8C8D", "gray_light": "#BDC3C7", "gray_dark": "#4A4A4A", "text": "#333333"}; CYC = [C["green"], C["orange"], C["teal"], C["coral"]]
    plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

LANGS = ["ko", "zh", "ja"]
LANG_NAME = {"ko": "Korean", "zh": "Chinese", "ja": "Japanese"}
YEARS = [2023, 2024, 2025, 2026]          # 2026은 1~9월(D30)
MIN_N = 10   # 투영 집합의 토픽별 건수가 이보다 작으면 lift를 보고하지 않는다

allc = pd.read_csv(DATA / f"corpus_llm_agent_all_{SUFFIX}.csv", dtype={"id": str})
cjk_signal = pd.read_csv(DATA / f"corpus_llm_agent_cjk_lang_{SUFFIX}.csv", dtype={"id": str})   # 키워드 신호 257편 + 사람 라벨(D36)
counts = json.loads(Path(sorted(glob.glob(str(VER / "corpus_counts_*.json")))[-1]).read_text())
YEAR_BASIS = counts["definitions"]["year_basis"]   # D31: "id" = arXiv id 접두(최초 제출 월)
LABEL_TARGET = "그 언어의 텍스트를 실제 연구 대상으로 다룸"; LABEL_MULTI = "다국어 실험의 한 언어로 포함"
cjk = cjk_signal[cjk_signal.cjk_lang].copy()       # 최종 대상 언어 집합 = 신호 ∧ 라벨 '실제 연구 대상'(D36)

# 정본 대조: 노트북이 읽은 자료가 정본 JSON과 같은지 확인한다
assert YEAR_BASIS == "id", "연도 기준이 D31(id 접두)이 아니다"
assert len(allc) == counts["pipeline"]["llm_agent_all"], "전체 건수가 정본과 다르다"
assert len(cjk_signal) == counts["pipeline"]["llm_agent_cjk_signal"], "대상 언어 신호 건수가 정본과 다르다"
assert len(cjk) == counts["pipeline"]["llm_agent_cjk_lang"], "대상 언어 집합 건수가 정본과 다르다"
assert (cjk_signal.lang_label != "").all(), "라벨 없는 신호 논문이 있다"
assert allc.groupby("year").size().to_dict() == {int(k): v for k, v in counts["llm_agent_by_year"].items()}
assert set(cjk.id) <= set(allc.id) and allc.id.is_unique
for l in LANGS:
    allc[f"lang_{l}"] = allc[f"lang_{l}"].astype(bool); allc[f"lang_{l}_signal"] = allc[f"lang_{l}_signal"].astype(bool)
assert {l: int(allc[f"lang_{l}"].sum()) for l in LANGS} == counts["cjk_lang_by_lang_total"]
N_ALL_BY_YEAR = {int(k): v for k, v in counts["pipeline"]["unique_by_year"].items()}   # 분모: 다섯 cs 분류의 고유 논문 수
print("정본 대조 통과: LLM 에이전트", len(allc), "| 한중일어 신호", len(cjk_signal), "| 대상 언어 집합(사람 라벨)", len(cjk), "| 연도 기준", YEAR_BASIS, "| 기간", counts["definitions"]["period"][:24])

## 1. 전처리 — spaCy 표제어 토큰

- 계산: 제목+초록을 소문자로 바꿔 spaCy로 표제어화(구문 분석·개체명 끔). 알파벳 세 글자 이상, 영어 불용어·도메인 불용어 제외
- 캐시: id → 토큰 목록을 jsonl로 저장해 재실행 때 다시 쓴다(자료가 바뀌면 캐시의 id 집합이 달라져 자동 재계산)


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
DOMAIN_STOP = {"paper", "propose", "proposed", "method", "methods", "result", "results", "show", "shows", "model", "models", "based",
               "approach", "approaches", "task", "tasks", "performance", "llm", "llms", "agent", "agents", "large", "language", "framework",
               "novel", "existing", "demonstrate", "experiments", "experimental", "study", "work", "using", "use", "data", "code", "available",
               "https", "github", "com", "et", "al", "paper", "present", "significantly", "propose", "introduce", "achieve", "outperform", "sota", "state", "art",
               "datum", "cod"}   # spaCy 표제어 특이값: data→datum, code(s)→cod 는 도메인 불용어(data, code)와 같은 말이다
STOP = set(ENGLISH_STOP_WORDS) | DOMAIN_STOP
cache = CACHE / f"lemmas_{SUFFIX}.jsonl"
lemmas = {}
if cache.exists():
    for line in cache.open():
        d = json.loads(line); lemmas[d["id"]] = d["t"]
need = [i for i in allc.id if i not in lemmas]
if need:
    import spacy
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts = (allc.set_index("id").loc[need].title.fillna("") + ". " + allc.set_index("id").loc[need].abstract.fillna("")).str.lower().tolist()
    with cache.open("a") as f:
        for i, doc in zip(need, nlp.pipe(texts, batch_size=128)):
            toks = [t.lemma_ for t in doc if t.is_alpha and len(t.lemma_) > 2 and not t.is_stop]   # 불용어는 캐시 뒤 단계에서 거른다(사전이 바뀌어도 캐시 재사용)
            lemmas[i] = toks; f.write(json.dumps({"id": i, "t": toks}) + "\n")
tokens = [[w for w in lemmas[i] if w not in STOP] for i in allc.id]   # 캐시는 걸러지기 전 상태일 수 있으므로 불용어를 여기서 다시 거른다
assert len(tokens) == len(allc) and len(set(allc.id) - set(lemmas)) == 0
print("문서", len(tokens), "건 | 평균 토큰", round(np.mean([len(t) for t in tokens]), 1), "| 캐시", cache.name, "(새로 계산", len(need), "건)")

## 2. 문서-단어 행렬

- 계산: 참고 코드와 같은 `CountVectorizer`. 토큰은 이미 표제어이므로 공백 분리만 한다. 문서 빈도 10 이상·50% 이하 단어만


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(tokenizer=str.split, preprocessor=None, lowercase=False, token_pattern=None, min_df=10, max_df=0.5)
dtm = cv.fit_transform([" ".join(t) for t in tokens]); vocab = np.array(cv.get_feature_names_out())
print("문서-단어 행렬", dtm.shape, "| 비어 있는 문서", int((dtm.sum(axis=1) == 0).sum()))

## 3. 토픽 수 선택 — c_v coherence와 보류 집합 perplexity

- 계산: K마다 `LatentDirichletAllocation(n_components=K, random_state=42, learning_method="batch", max_iter=20)`을 학습 집합(90%)에 적합. 토픽별 상위 10단어로 c_v coherence(gensim)를, 보류 10%로 perplexity를 구한다
- 선택 규칙: coherence 최대 K. 두 K가 0.01 안이면 작은 K. perplexity는 참고로 함께 보인다
- 출력: `analysis/output/tables/lda_model_selection.csv`, `analysis/output/figures/fig2a_lda_model_selection.png`


In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
K_GRID = [10, 15, 20, 25, 30]
idx_tr, idx_te = train_test_split(np.arange(dtm.shape[0]), test_size=0.1, random_state=42)
gdict = Dictionary(tokens)
def top_words(model, n=10):
    return [[vocab[j] for j in comp.argsort()[::-1][:n]] for comp in model.components_]
sel, models = [], {}
for k in K_GRID:
    m = LatentDirichletAllocation(n_components=k, random_state=42, learning_method="batch", max_iter=20, n_jobs=-1).fit(dtm[idx_tr])
    coh = CoherenceModel(topics=top_words(m), texts=tokens, dictionary=gdict, coherence="c_v", topn=10, processes=1).get_coherence()
    sel.append({"K": k, "c_v": round(coh, 4), "perplexity_heldout": round(m.perplexity(dtm[idx_te]), 1)}); models[k] = m
    print(sel[-1])
sel = pd.DataFrame(sel).set_index("K"); sel.to_csv(TAB / "lda_model_selection.csv")
best = int(sel.c_v.idxmax())
for k in sel.index:
    if sel.c_v[best] - sel.c_v[k] <= 0.01 and k < best:
        best = int(k); break
K = best; print("선택한 K =", K)
fig, a1 = plt.subplots(figsize=(5, 2.8)); a1.plot(sel.index, sel.c_v, "o-", label="c_v coherence"); a1.set_xlabel("K"); a1.set_ylabel("c_v")
a2 = a1.twinx(); a2.plot(sel.index, sel.perplexity_heldout, "s--", color="gray", label="held-out perplexity"); a2.set_ylabel("perplexity")
a1.axvline(K, color="#C0392B", lw=1, ls=":"); a1.set_title("LDA model selection"); h1, l1 = a1.get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels(); a1.legend(h1 + h2, l1 + l2, fontsize=7, frameon=False)
plt.tight_layout(); plt.savefig(FIG / "fig2a_lda_model_selection.png", dpi=160, bbox_inches="tight"); plt.show()

## 4. 최종 모델 — 전체 문서로 다시 적합, 상위 단어, 문서 배정

- 계산: 선택한 K로 전체 문서에 적합. 토픽별 상위 12단어. 문서마다 확률 최대 토픽을 배정하고 최대 확률도 남긴다
- 라벨: 상위 두 단어(예: `gui·mobile`). 사람이 지은 이름은 쓰지 않는다(D20)
- 출력: `analysis/output/tables/lda_topics_K{K}.csv`, `lda_doc_topics.csv`


In [ ]:
lda = LatentDirichletAllocation(n_components=K, random_state=42, learning_method="batch", max_iter=25, n_jobs=-1).fit(dtm)
theta = lda.transform(dtm)
allc["lda_topic"] = theta.argmax(axis=1); allc["lda_prob"] = theta.max(axis=1)
tw = top_words(lda, 12)
lda_topics = pd.DataFrame({"topic": range(K), "label": [f"{w[0]}·{w[1]}" for w in tw], "top_words": [", ".join(w) for w in tw],
                           "n_docs": pd.Series(allc.lda_topic).value_counts().reindex(range(K), fill_value=0).values})
lda_topics["share_pct"] = (100 * lda_topics.n_docs / len(allc)).round(2)
lda_topics.to_csv(TAB / f"lda_topics_K{K}.csv", index=False)
allc[["id", "year", "month", "lda_topic", "lda_prob"]].to_csv(TAB / "lda_doc_topics.csv", index=False)
lda_label = lambda t: lda_topics.label[t]
pd.set_option("display.max_colwidth", 160); lda_topics.sort_values("n_docs", ascending=False)

## 5. pyLDAvis — 토픽 간 거리와 단어 분포(대화형)

- 계산: 참고 코드와 같은 `pyLDAvis.lda_model.prepare(lda, dtm, cv)`. HTML로 저장한다(원고에는 넣지 않고 탐색·발표용)
- 출력: `analysis/output/figures/lda_pyldavis_K{K}.html`


In [ ]:
try:
    import pyLDAvis, pyLDAvis.lda_model
    vis = pyLDAvis.lda_model.prepare(lda, dtm, cv, mds="mmds", sort_topics=False)
    pyLDAvis.save_html(vis, str(FIG / f"lda_pyldavis_K{K}.html")); print("saved", FIG / f"lda_pyldavis_K{K}.html")
except Exception as e:
    print("pyLDAvis 생략:", e)

## 6. 투영 — 한중일어 대상 집합의 토픽 분포 대 세계 전체

- 계산: 토픽별로 세계 전체 비중, 언어 집합의 관측 편수, 기대 편수(= 세계 비중 × 집합 크기), lift(관측/기대, 10건 미만은 NaN). 두 번째 기준선: 한중일 이외 언어명이나 multilingual/cross-lingual을 적은 세계 논문의 토픽 비중 × 집합 크기(exp_lm). 토픽별 언어명 언급률(lang_mention_pct_of_topic)도 남긴다
- 순열 검정: 집합 라벨을 2,000회 무작위 배정해 "차이 없음"일 때의 JSD(집합 대 나머지) 분포를 만들고 관측 JSD와 비교(p)
- 출력: `analysis/output/tables/lda_project_lang.csv`, `analysis/output/figures/fig2b_lda_obs_vs_exp_lang.png`


In [ ]:
from scipy.spatial.distance import jensenshannon
T = list(range(K)); share_all = allc.lda_topic.value_counts(normalize=True).reindex(T, fill_value=0)
groups = {l: allc[f"lang_{l}"].values for l in LANGS}
# 두 번째 기준선(리뷰 2026-10-14): "한중일 이외의 언어명 또는 multilingual/cross-lingual을 적은 세계 논문"의 토픽 분포.
# 언어명이 적히는 논문 자체가 평가·대화 토픽에 몰리므로, 이 기준선으로 비교하면 한중일 집합만의 결손과 '그 토픽 논문은 언어명을 안 적는다'는 표본틀의 성질을 가른다.
OTHER_LANGS = ["english", "spanish", "french", "german", "italian", "portuguese", "russian", "arabic", "hindi", "bengali", "urdu", "vietnamese", "thai", "indonesian", "malay", "tagalog", "filipino", "turkish", "persian", "farsi", "hebrew", "greek", "dutch", "swedish", "danish", "norwegian", "finnish", "polish", "czech", "hungarian", "romanian", "ukrainian", "swahili", "yoruba", "hausa", "amharic", "tamil", "telugu", "marathi", "gujarati", "punjabi", "kannada", "malayalam", "nepali", "sinhala", "burmese", "khmer", "lao", "mongolian", "tibetan", "uyghur", "kazakh", "uzbek", "basque", "catalan", "galician", "welsh", "irish", "icelandic", "estonian", "latvian", "lithuanian", "slovak", "slovenian", "croatian", "serbian", "bulgarian", "macedonian", "albanian", "georgian", "armenian", "azerbaijani", "afrikaans", "zulu", "xhosa", "igbo", "somali", "latin", "sanskrit", "hokkien", "cantonese", "javanese", "sundanese", "cebuano", "hmong", "quechua", "maori", "hawaiian"]
LM_RE = re.compile(r"\b(" + "|".join(OTHER_LANGS) + r")\b|multi-?lingual|cross-?lingual")
_txt = (allc.title.fillna("") + " " + allc.abstract.fillna("")).str.lower()
allc["lang_mention"] = _txt.map(lambda x: bool(LM_RE.search(x)))
allc["cjk_signal"] = allc["cjk_signal"].astype(bool)
lm = allc[allc.lang_mention & ~allc.cjk_signal]; share_lm = lm.lda_topic.value_counts(normalize=True).reindex(T, fill_value=0)   # 한중일 신호 논문은 기준선에서 뺀다(모의 심사: 부분 동어반복)
print(f"언어명(한중일 제외)·multilingual 언급 논문(한중일 신호 없는 것) {len(lm):,}편({100*len(lm)/len(allc):.1f}%) — 두 번째 기준선")
cols = {"label": [lda_label(t) for t in T], "n_all": allc.lda_topic.value_counts().reindex(T, fill_value=0).values, "share_all_pct": (100 * share_all).round(2).values,
        "n_lang_mention": lm.lda_topic.value_counts().reindex(T, fill_value=0).values, "lang_mention_pct_of_topic": (100 * lm.lda_topic.value_counts().reindex(T, fill_value=0) / allc.lda_topic.value_counts().reindex(T)).round(2).values}
for l, m in groups.items():
    obs = allc[m].lda_topic.value_counts().reindex(T, fill_value=0); exp = share_all * m.sum(); exp_lm = share_lm * m.sum()
    cols[f"obs_{l}"] = obs.values; cols[f"exp_{l}"] = exp.round(1).values; cols[f"lift_{l}"] = (obs / exp).round(2).where(obs >= MIN_N).values
    cols[f"exp_lm_{l}"] = exp_lm.round(1).values; cols[f"ratio_lm_{l}"] = (obs / exp_lm.replace(0, np.nan)).round(2).values
t_lang = pd.DataFrame(cols, index=pd.Index(T, name="topic")); t_lang.to_csv(TAB / "lda_project_lang.csv")
from scipy.stats import spearmanr
print("토픽별 언어명 언급률 대 중국어 관측/기대 Spearman:", spearmanr(t_lang.lang_mention_pct_of_topic, t_lang.obs_zh / t_lang.exp_zh.replace(0, np.nan), nan_policy="omit"))
rng = np.random.default_rng(42); top_arr = allc.lda_topic.values
def jsd_vs_rest(mask):
    pa = np.bincount(top_arr[mask], minlength=K) / max(mask.sum(), 1); pb = np.bincount(top_arr[~mask], minlength=K) / max((~mask).sum(), 1)
    return float(jensenshannon(pa, pb, base=2) ** 2)
perm = {}
for l, m in groups.items():
    n = int(m.sum()); point = jsd_vs_rest(m)
    null = np.array([jsd_vs_rest(np.isin(np.arange(len(m)), rng.choice(len(m), n, replace=False))) for _ in range(2000)])
    perm[f"lang:{l}"] = {"n": n, "jsd_vs_rest": round(point, 4), "null_mean": round(float(null.mean()), 4), "null_q95": round(float(np.quantile(null, 0.95)), 4), "p_perm": round(float((np.sum(null >= point) + 1) / 2001), 4)}
print(json.dumps(perm, indent=1)); display(t_lang.sort_values("n_all", ascending=False))
fig, axes = plt.subplots(1, 3, figsize=(13, 0.3 * K + 1.8), sharey=True); order = t_lang.sort_values("n_all", ascending=False).index
for ax, l in zip(axes, LANGS):
    sub = t_lang.loc[order]; y = np.arange(K)
    ax.barh(y - 0.27, sub[f"obs_{l}"], 0.27, color=C["green"], label="observed", zorder=3); ax.barh(y, sub[f"exp_{l}"], 0.27, color=C["gray_light"], label="expected: world share × n", zorder=3)
    ax.barh(y + 0.27, sub[f"exp_lm_{l}"], 0.27, color=C["orange_light"], label="expected: language-mentioning papers' share × n", zorder=3)
    xmax = max(sub[f"obs_{l}"].max(), sub[f"exp_{l}"].max(), sub[f"exp_lm_{l}"].max())
    for i, t in enumerate(order): ax.text(max(sub.loc[t, f"obs_{l}"], sub.loc[t, f"exp_{l}"], sub.loc[t, f"exp_lm_{l}"]) + 0.02 * xmax, i, f"{int(sub.loc[t, f'obs_{l}'])} ({sub.loc[t, f'exp_{l}']:.1f} | {sub.loc[t, f'exp_lm_{l}']:.1f})", va="center", fontsize=6.5, color=C["text"])
    for cut in (7.5, 16.5): ax.axhline(cut, color=C["gray"], lw=0.7, ls=":")
    ax.set_yticks(y); ax.set_yticklabels([f"{i + 1}. {sub.label[t]}" for i, t in enumerate(order)], fontsize=7.5); ax.invert_yaxis(); ax.set_title(f"{LANG_NAME[l]} (n={int(groups[l].sum())})"); ax.set_xlabel("papers"); ax.grid(axis="y", visible=False)
    ax.set_xlim(0, xmax * 1.4)
axes[1].legend(loc="lower right"); fig.suptitle("Observed papers per topic vs two expected baselines (rank = Table 2; dotted lines = top 8 / middle 9 / bottom 8)", y=1.0, fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG / "fig2b_lda_obs_vs_exp_lang.png", dpi=200, bbox_inches="tight"); plt.show()

## 7. 연도별 토픽 점유율(세계 전체)

- 계산: 연도 × 토픽 점유율(열 합 1). 2026은 1~9월. 2026 대 2023 차이(퍼센트포인트)로 정렬
- 출력: `analysis/output/tables/lda_year_share.csv`, `analysis/output/figures/fig2c_lda_year_share.png`


In [ ]:
yt = pd.crosstab(allc.lda_topic, allc.year).reindex(index=T, columns=YEARS, fill_value=0)
yt_share = (100 * yt / yt.sum()).round(2); yt_share["pp_2026_vs_2023"] = (yt_share[2026] - yt_share[2023]).round(2)
yt_share.insert(0, "label", [lda_label(t) for t in T]); yt_share.to_csv(TAB / "lda_year_share.csv")
display(yt_share.sort_values("pp_2026_vs_2023", ascending=False))
M = yt_share.sort_values("pp_2026_vs_2023", ascending=False); fig, ax = plt.subplots(figsize=(6, 0.28 * K + 1.2))
im = ax.imshow(M[YEARS].values, cmap="Blues", aspect="auto"); ax.set_xticks(range(4)); ax.set_xticklabels([str(y) if y != 2026 else "2026\n(Jan-Sep)" for y in YEARS])
ax.set_yticks(range(K)); ax.set_yticklabels([f"T{t} {M.label[t]}" for t in M.index], fontsize=7)
for i in range(K):
    for j in range(4): ax.text(j, i, f"{M[YEARS].values[i, j]:.1f}", ha="center", va="center", fontsize=6.5, color="white" if M[YEARS].values[i, j] > M[YEARS].values.max() * 0.6 else "black")
plt.colorbar(im, ax=ax, fraction=0.03, label="share of year (%)"); ax.set_title("LDA topic share by year (world)"); plt.tight_layout(); plt.savefig(FIG / "fig2c_lda_year_share.png", dpi=160, bbox_inches="tight"); plt.show()

## 7b. 세계 상위·중위·하위 토픽과 언어 집합의 자리 (기간 2023-01 ~ 2026-09 하나로)

- 계산: 토픽을 세계 전체 편수(2023~2026-09 합) 순으로 세워 상위(1~8위)·중위(9~17위)·하위(18~25위)로 나눈다. 칸마다 총 편수와 네 해(2023/2024/2025/2026)의 편수를 그대로 적는다. 세계는 연도별 점유율(열 합 100%)과 그 네 점에 맞춘 직선 기울기(pp/년)를 한 열로 둔다 — 오름(+0.5pp/년 이상), 내림(−0.5 이하), 유지.
- 언어 집합(한국어·중국어·일본어)은 총 관측 편수, 네 해의 편수, 기대 편수(세계 비중 × 집합 크기), 그리고 관측/기대 판정(기대의 1.5배 초과 = 많음, 절반 미만 = 적음)을 둔다. 집합이 작아 연도별 점유율은 쓰지 않는다.
- "비어 있는 자리" = 세계 상위 또는 오르는 토픽인데 그 언어 집합의 관측이 기대의 절반 미만.
- 입력: 7절의 `yt`(연도 × 토픽 편수), 6절의 `t_lang`
- 출력: `analysis/output/tables/lda_world_topic_table.csv`, `lda_world_topic_table.md`(원고용), `analysis/output/figures/fig2d_lda_world_topics.png`
- 주의: 2026은 9개월치이므로 편수가 아니라 점유율로 기울기를 잰다. 2023년은 689편(토픽당 평균 28편)이라 점유율이 거칠다. 토픽 경계는 seed에 따라 흔들리므로(9절) 세부 순위보다 상·중·하위 묶음으로 읽는다.


In [ ]:
yt_pct = 100 * yt / yt.sum()
wd = pd.DataFrame({"label": [lda_label(t) for t in T], "world_total": yt.loc[T, YEARS].sum(axis=1).values}, index=pd.Index(T, name="topic"))
wd["world_share_pct"] = (100 * wd.world_total / wd.world_total.sum()).round(2)
for y in YEARS:
    wd[f"world_n_{y}"] = yt.loc[T, y].values; wd[f"world_share_{y}"] = yt_pct.loc[T, y].round(1).values
wd["world_slope_pp_per_year"] = [round(float(np.polyfit(YEARS, yt_pct.loc[t, YEARS].values.astype(float), 1)[0]), 2) for t in T]
wd["world_trend"] = np.where(wd.world_slope_pp_per_year >= 0.5, "rising", np.where(wd.world_slope_pp_per_year <= -0.5, "falling", "flat"))
wd = wd.sort_values("world_total", ascending=False); wd["world_rank"] = range(1, K + 1)
wd["world_tier"] = np.where(wd.world_rank <= 8, "top", np.where(wd.world_rank <= 17, "middle", "bottom"))
for l in LANGS:
    m = allc[f"lang_{l}"].values
    for y in YEARS:
        wd[f"{l}_n_{y}"] = [int(((allc.lda_topic == t) & m & (allc.year == y).values).sum()) for t in wd.index]
    wd[f"{l}_obs"] = t_lang.loc[wd.index, f"obs_{l}"].values; wd[f"{l}_exp"] = t_lang.loc[wd.index, f"exp_{l}"].values
    wd[f"{l}_gap"] = np.where(wd[f"{l}_obs"] > wd[f"{l}_exp"] * 1.5, "over", np.where(wd[f"{l}_obs"] < wd[f"{l}_exp"] * 0.5, "under", "near"))
    wd[f"{l}_exp_lm"] = t_lang.loc[wd.index, f"exp_lm_{l}"].values
    wd[f"{l}_gap_lm"] = np.where(wd[f"{l}_exp_lm"] < 1, "n/a", np.where(wd[f"{l}_obs"] > wd[f"{l}_exp_lm"] * 1.5, "over", np.where(wd[f"{l}_obs"] < wd[f"{l}_exp_lm"] * 0.5, "under", "near")))
wd["lang_mention_pct"] = t_lang.loc[wd.index, "lang_mention_pct_of_topic"].values
wd["empty_seat"] = [", ".join(LANG_NAME[l] for l in LANGS if (r.world_tier == "top" or r.world_trend == "rising") and getattr(r, f"{l}_gap") == "under") for r in wd.itertuples()]
wd.to_csv(TAB / "lda_world_topic_table.csv")
print("세계 상위 8:", ", ".join(f"{r.label}({int(r.world_total):,}, {r.world_slope_pp_per_year:+.2f}pp/년)" for r in wd[wd.world_tier == "top"].itertuples()))
print("세계 하위 8:", ", ".join(f"{r.label}({int(r.world_total):,}, {r.world_slope_pp_per_year:+.2f}pp/년)" for r in wd[wd.world_tier == "bottom"].itertuples()))
print("비어 있는 자리(상위 또는 오르는 토픽에서 기대의 절반 미만):"); print(wd[wd.empty_seat != ""][["label", "world_tier", "world_trend", "empty_seat"]].to_string())

# 원고용 Markdown 표
def ycounts(r, s): return "/".join(str(int(r[f"{s}_n_{y}"])) for y in YEARS)
def wcell(r): return f"**{int(r['world_total']):,}** ({r['world_share_pct']:.1f}%) · {ycounts(r, 'world')} · 점유율 " + "/".join(f"{r[f'world_share_{y}']:.1f}" for y in YEARS) + f"% · 기울기 {r['world_slope_pp_per_year']:+.2f}pp/년"
def lcell(r, l): return f"**{int(r[f'{l}_obs'])}** · {ycounts(r, l)} · 기대 {r[f'{l}_exp']:.1f}" + (" · 많음" if r[f"{l}_gap"] == "over" else " · 적음" if r[f"{l}_gap"] == "under" else "")
sizes = {"world": [int((allc.year == y).sum()) for y in YEARS], **{l: [int((allc[f"lang_{l}"] & (allc.year == y)).sum()) for y in YEARS] for l in LANGS}}
head = ("| 순위 | 토픽 | 세계 (총 " + f"{len(allc):,}; " + "/".join(f"{n:,}" for n in sizes["world"]) + ") | " + " | ".join(f"{LANG_NAME[l]} (총 {sum(sizes[l])}; " + "/".join(str(n) for n in sizes[l]) + ")" for l in LANGS) + " |\n|---|---|---|---|---|---|\n")
def block(df): return head + "".join(f"| {int(r['world_rank'])} | {r['label']} | {wcell(r)} | " + " | ".join(lcell(r, l) for l in LANGS) + " |\n" for _, r in df.iterrows())
md = "칸 읽는 법: **총 편수(2023-01~2026-09)** (세계 비중 %) · 연도별 편수 2023/2024/2025/2026(1~9월) · (세계만) 연도별 점유율 %와 네 점 직선 기울기 pp/년 · (언어 집합) 기대 = 세계 비중 × 집합 크기, 많음 = 기대의 1.5배 초과, 적음 = 기대의 절반 미만. 네 집합 모두 같은 자료·같은 기간.\n\n"
for tier, name in (("top", "세계 상위 토픽(1~8위)"), ("middle", "세계 중위 토픽(9~17위)"), ("bottom", "세계 하위 토픽(18~25위)")):
    md += f"### {name}\n\n" + block(wd[wd.world_tier == tier]) + "\n"
(TAB / "lda_world_topic_table.md").write_text(md); print("saved", TAB / "lda_world_topic_table.md")

fig, ax = plt.subplots(figsize=(8, 0.3 * K + 1.2)); y = np.arange(K)
ax.barh(y, wd.world_share_pct, color=["#C0392B" if d == "rising" else "#2E86C1" if d == "falling" else "#B0B7C3" for d in wd.world_trend])
ax.set_yticks(y); ax.set_yticklabels([f"{int(r.world_rank)}. {r.label}" for r in wd.itertuples()], fontsize=7); ax.invert_yaxis(); ax.set_xlabel("world share of LLM-agent papers, 2023-01 to 2026-09 (%)")
for i, r in enumerate(wd.itertuples()):
    ax.text(r.world_share_pct + 0.1, i, f"{r.world_slope_pp_per_year:+.1f}pp/yr   " + " ".join(f"{LANG_NAME[l][:2]}:{getattr(r, f'{l}_gap')[0].upper()}" for l in LANGS), va="center", fontsize=6.5, color="gray")
for cut in (7.5, 16.5): ax.axhline(cut, color="k", lw=0.6, ls=":")
ax.set_title("World topics by size (bar colour = trend: red rising, blue falling); right: slope and Ko/Ch/Ja gap (O over, U under, N near)", fontsize=8.5)
plt.tight_layout(); plt.savefig(FIG / "fig2d_lda_world_topics.png", dpi=160, bbox_inches="tight"); plt.show()

## 8. 언어 집합의 상위 단어(문서 빈도)와 세계 대비 비율

- 계산: 언어 집합 안에서 표제어의 문서 빈도(%)와 세계 전체 문서 빈도(%)의 비율. 집합에서 5편 이상 나타난 단어만. 토픽 이름 대신 자료에서 나온 단어로 집합의 성격을 적는다(D20)
- 출력: `analysis/output/tables/lda_lang_subset_keywords.csv`


In [ ]:
docs_sets = [set(t) for t in tokens]; n_all = len(docs_sets)
df_all = pd.Series([w for d in docs_sets for w in d]).value_counts()
rows = []
for l, m in groups.items():
    sub = [docs_sets[i] for i in np.where(m)[0]]; n = len(sub)
    df_sub = pd.Series([w for d in sub for w in d]).value_counts()
    df_sub = df_sub[df_sub >= 5]
    r = pd.DataFrame({"lang": LANG_NAME[l], "word": df_sub.index, "doc_pct_lang": (100 * df_sub / n).round(1).values, "doc_pct_world": (100 * df_all.reindex(df_sub.index).fillna(0) / n_all).round(2).values})
    r["ratio"] = (r.doc_pct_lang / r.doc_pct_world.replace(0, np.nan)).round(1); rows.append(r.sort_values("ratio", ascending=False).head(20))
subset_kw = pd.concat(rows); subset_kw.to_csv(TAB / "lda_lang_subset_keywords.csv", index=False)
for l in LANGS: print(LANG_NAME[l], "→", ", ".join(f"{w}({d:.0f}%)" for w, d in subset_kw[subset_kw.lang == LANG_NAME[l]][["word", "doc_pct_lang"]].head(15).values))

## 9. 안정성 — seed 3개

- 계산: random_state 43/44로 같은 K를 다시 적합하고 토픽별 상위 10단어 집합의 최대 Jaccard. 평균 0.5 미만 토픽은 "불안정"으로 표시


In [ ]:
base_sets = [set(w) for w in top_words(lda, 10)]; stab = []
for seed in [43, 44]:
    m = LatentDirichletAllocation(n_components=K, random_state=seed, learning_method="batch", max_iter=25, n_jobs=-1).fit(dtm)
    other = [set(w) for w in top_words(m, 10)]; stab.append([max(len(a & b) / len(a | b) for b in other) for a in base_sets])
stab = pd.DataFrame(stab, index=["seed43", "seed44"], columns=[f"T{t}" for t in T]).T
stab["mean_jaccard"] = stab.mean(axis=1).round(2); stab["stable"] = stab.mean_jaccard >= 0.5
print("안정 토픽", int(stab.stable.sum()), "/", K); stab

## 9b. 시드별 연도 점유율 기울기 — 오름·내림 방향이 시드를 바꿔도 반복되는가(리뷰 2026-10-14)

- 계산: 시드 43·44 모델의 문서 배정으로 연도 점유율 기울기(pp/년)를 다시 구하고, 각 기준 토픽(seed 42)에 상위 10단어 자카드가 가장 큰 시드 토픽을 대응시켜 기울기 부호를 비교한다. 세 모델 모두 +0.5pp/년 이상 또는 모두 −0.5pp/년 이하이면 "반복"으로 표시한다.
- 2026년 1위 토픽(benchmark·score)의 월별 점유율도 남긴다(1월→9월).
- 출력: `analysis/output/tables/lda_slope_by_seed.csv`


In [ ]:
def year_slopes(model):
    assign = model.transform(dtm).argmax(axis=1); ct = pd.crosstab(assign, allc.year.values).reindex(columns=YEARS, fill_value=0); pct = 100 * ct / ct.sum()
    return {t: round(float(np.polyfit(YEARS, pct.loc[t, YEARS].values.astype(float), 1)[0]), 2) if t in pct.index else np.nan for t in range(K)}, model
base_slope = {t: float(wd.loc[t, "world_slope_pp_per_year"]) for t in T}; rows = []
seed_models = {}
for seed in [43, 44]:
    m = LatentDirichletAllocation(n_components=K, random_state=seed, learning_method="batch", max_iter=25, n_jobs=-1).fit(dtm); seed_models[seed] = m
    sl, _ = year_slopes(m); other = [set(w) for w in top_words(m, 10)]
    for t in T:
        j = int(np.argmax([len(base_sets[t] & o) / len(base_sets[t] | o) for o in other])); rows.append({"topic": t, "seed": seed, "matched": j, "jaccard": round(len(base_sets[t] & other[j]) / len(base_sets[t] | other[j]), 2), "slope": sl[j]})
SS = pd.DataFrame(rows).pivot(index="topic", columns="seed", values=["slope", "jaccard"]); SS.columns = [f"{a}_{b}" for a, b in SS.columns]
SS["slope_42"] = [base_slope[t] for t in SS.index]; SS["label"] = [lda_label(t) for t in SS.index]
sign = lambda x: "+" if x >= 0.5 else "-" if x <= -0.5 else "0"
SS["direction_42_43_44"] = [sign(r.slope_42) + sign(r.slope_43) + sign(r.slope_44) for r in SS.itertuples()]
SS["repeated"] = SS.direction_42_43_44.isin(["+++", "---"]); SS = SS.sort_values("slope_42", ascending=False); SS.to_csv(TAB / "lda_slope_by_seed.csv")
print("세 시드 모두 같은 방향(|기울기| ≥ 0.5pp/년):"); print(SS[SS.repeated][["label", "slope_42", "slope_43", "slope_44", "jaccard_43", "jaccard_44"]].to_string())
top_t = int(wd.index[0]); m26 = allc[allc.year == 2026]; monthly_top = (100 * m26.groupby("month").lda_topic.apply(lambda s: (s == top_t).mean())).round(1)
print(f"\n1위 토픽 {lda_label(top_t)} 2026년 월별 점유율(%):", monthly_top.to_dict())
slope_by_seed = {str(t): {"label": r.label, "slope_42": r.slope_42, "slope_43": r.slope_43, "slope_44": r.slope_44, "jaccard_43": r.jaccard_43, "jaccard_44": r.jaccard_44, "direction": r.direction_42_43_44, "repeated": bool(r.repeated)} for t, r in SS.iterrows()}

## 10. 정본 저장

- 저장: K 선택표, 토픽 단어·라벨·건수, 투영 표(관측·기대·lift), 순열 검정, 연도 점유율, 집합 상위 단어, 안정성 → `verification/topics_lda_YYYYMMDD.json`


In [ ]:
out = {"generated": TODAY, "notebook": "analysis/notebooks/02_lda.ipynb", "period": counts["definitions"]["period"], "year_basis": YEAR_BASIS,
       "n_docs": int(len(allc)), "vocab_size": int(len(vocab)), "vectorizer": {"min_df": 10, "max_df": 0.5, "tokens": "spaCy en_core_web_sm lemma, alpha, len>2, stopwords removed"},
       "domain_stopwords": sorted(DOMAIN_STOP), "model_selection": {str(k): {c: float(v) for c, v in r.items()} for k, r in sel.iterrows()}, "K": int(K),
       "lda_params": {"random_state": 42, "learning_method": "batch", "max_iter": 25},
       "topics": {int(r.topic): {"label": r.label, "top_words": r.top_words.split(", "), "n_docs": int(r.n_docs), "share_pct": float(r.share_pct)} for r in lda_topics.itertuples()},
       "project_lang": {str(t): {c: (None if pd.isna(v) else (float(v) if isinstance(v, (float, np.floating)) else (int(v) if isinstance(v, (int, np.integer)) else v))) for c, v in r.items()} for t, r in t_lang.iterrows()},
       "permutation": perm,
       "year_share_pct": {str(t): {str(y): float(yt_share.loc[t, y]) for y in YEARS} for t in T},
       "world_topic_table": {str(t): {c: (None if pd.isna(v) else (v if isinstance(v, str) else float(v))) for c, v in r.items()} for t, r in wd.iterrows()},
       "lang_subset_keywords": {LANG_NAME[l]: subset_kw[subset_kw.lang == LANG_NAME[l]][["word", "doc_pct_lang", "doc_pct_world", "ratio"]].to_dict(orient="records") for l in LANGS},
       "stability_mean_jaccard": {k: float(v) for k, v in stab.mean_jaccard.items()},
       "slope_by_seed": slope_by_seed, "top_topic_2026_monthly_share_pct": {str(k): float(v) for k, v in monthly_top.items()},
       "lang_mention_baseline": {"definition": "title+abstract contains a non-CJK language name or multilingual/cross-lingual, and no CJK language-name signal", "n": int(len(lm)), "share_pct": round(100 * len(lm) / len(allc), 2)}}
p = VER / f"topics_lda_{TODAY}.json"; p.write_text(json.dumps(out, ensure_ascii=False, indent=1)); print("saved", p)